In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
pip install alpaca-trade-api

In [41]:
pip install polygon-api-client

In [42]:
import alpaca_trade_api as tradeapi
from polygon import RESTClient
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
import tensorflow as tf
from tensorflow.keras.models import load_model
import joblib
import numpy as np

In [43]:
ALPACA_API_KEY = "AK1RX6F8W6QX207XPLDF"
ALPACA_SECRET_KEY = "WaPoTTxkQBGzC51LajCdyw8Pl6svbINa9eDu9TMK"
ALPACA_BASE_URL = "https://api.alpaca.markets"

POLYGON_API_KEY = "Gkm8qM_seLdMVw3YRiSBODhwwmpPPpn4"

In [44]:
# Inicializa a API Alpaca com as credenciais lidas do arquivo
api = tradeapi.REST(ALPACA_API_KEY, ALPACA_SECRET_KEY, ALPACA_BASE_URL, api_version='v2')

In [45]:
clock = api.get_clock()

In [46]:
clock

Clock({   'is_open': True,
    'next_close': '2025-05-09T16:00:00-04:00',
    'next_open': '2025-05-12T09:30:00-04:00',
    'timestamp': '2025-05-09T14:20:27.561349746-04:00'})

In [47]:
client = RESTClient(POLYGON_API_KEY)

In [48]:
status = client.get_market_status()

In [49]:
status

MarketStatus(after_hours=False, currencies=MarketCurrencies(crypto='open', fx='open'), early_hours=False, exchanges=MarketExchanges(nasdaq='open', nyse='open', otc='open'), indicesGroups=MarketIndices(s_and_p='open', societe_generale='open', cgi='open', msci='open', ftse_russell='open', mstar='open', mstarc='open', cccy='open', nasdaq='open', dow_jones='open'), market='open', server_time='2025-05-09T14:20:27-04:00')

In [50]:
if status.market == "open":
    print("The market is open regular hours.")
elif status.early_hours:
    print("The market is open pre hours.")
elif status.after_hours:
    print("The market is open after hours.")
else:
    print("The market is closed.")

The market is open regular hours.


In [51]:
delayed_safe_df = 5

now   = datetime.now(timezone.utc)
start = (now - timedelta(days=delayed_safe_df)).isoformat()

print("Now (UTC):", now)
print("Start (UTC):", start)

Now (UTC): 2025-05-09 18:20:27.865724+00:00
Start (UTC): 2025-05-04T18:20:27.865724+00:00


In [52]:
symbol = "AAPL"
timeframe = "5Min"
data_source = "sip"
dataset_size = 36

In [53]:
# Fetch the historical data
bars = api.get_bars(
    symbol,
    timeframe,
    start,
    feed=data_source
).df.tail(dataset_size)

In [54]:
bars

,close,high,low,trade_count,open,volume,vwap
timestamp,,,,,,,
2025-05-09 15:10:00+00:00,198.8900,198.9050,198.3100,5045,198.8800,433739,198.701376
2025-05-09 15:15:00+00:00,198.8200,199.1800,198.6900,4534,198.8900,401799,198.938096
2025-05-09 15:20:00+00:00,198.3900,198.8397,198.3558,4775,198.8200,334010,198.546657
2025-05-09 15:25:00+00:00,198.4300,198.5200,198.2000,4234,198.3801,328932,198.374568
2025-05-09 15:30:00+00:00,198.5501,198.5700,198.1400,4738,198.4250,332396,198.287617
2025-05-09 15:35:00+00:00,198.3500,198.6640,198.3100,4100,198.5700,266078,198.496335
2025-05-09 15:40:00+00:00,198.5279,198.5650,198.2300,3184,198.3500,203919,198.434810
2025-05-09 15:45:00+00:00,198.6836,198.7804,198.3600,4153,198.5300,264342,198.604140
2025-05-09 15:50:00+00:00,198.7000,199.0100,198.5600,3985,198.6700,263119,198.870335


In [55]:
data = bars[['vwap', 'trade_count']]

In [56]:
data

,vwap,trade_count
timestamp,,
2025-05-09 15:10:00+00:00,198.701376,5045
2025-05-09 15:15:00+00:00,198.938096,4534
2025-05-09 15:20:00+00:00,198.546657,4775
2025-05-09 15:25:00+00:00,198.374568,4234
2025-05-09 15:30:00+00:00,198.287617,4738
2025-05-09 15:35:00+00:00,198.496335,4100
2025-05-09 15:40:00+00:00,198.434810,3184
2025-05-09 15:45:00+00:00,198.604140,4153
2025-05-09 15:50:00+00:00,198.870335,3985


In [57]:
scaler = joblib.load("/content/drive/MyDrive/AI Financial Analisys/Summer Project/Live Trading/minmax_ds=sip+s=AAPL+mp=False+sd=2016-01-1+ed=2024-12-30+tf=5Min.pkl")

In [58]:
X_VWAP = data[['vwap']].to_numpy()

X_VWAP_scaled = scaler.transform(X_VWAP)

In [59]:
X_VWAP_scaled

array([[0.25685254],
       [0.25740877],
       [0.25648898],
       [0.25608462],
       [0.2558803 ],
       [0.25637074],
       [0.25622617],
       [0.25662406],
       [0.25724955],
       [0.25679825],
       [0.25614989],
       [0.25560859],
       [0.25553796],
       [0.25623993],
       [0.25607776],
       [0.25573347],
       [0.25551924],
       [0.25553202],
       [0.25513876],
       [0.25496083],
       [0.25513994],
       [0.25552389],
       [0.25529226],
       [0.255309  ],
       [0.25593748],
       [0.25577252],
       [0.25534987],
       [0.25578988],
       [0.25537169],
       [0.25547444],
       [0.2560988 ],
       [0.25676275],
       [0.25656806],
       [0.25686669],
       [0.25707899],
       [0.25725818]])

In [60]:
X_Trade_Count = data[['trade_count']].to_numpy()

In [61]:
X_Trade_Count

array([[5045],
       [4534],
       [4775],
       [4234],
       [4738],
       [4100],
       [3184],
       [4153],
       [3985],
       [3096],
       [4292],
       [3606],
       [4317],
       [3825],
       [2644],
       [2713],
       [2912],
       [3887],
       [3062],
       [3946],
       [3563],
       [3321],
       [3894],
       [3739],
       [3331],
       [3123],
       [7344],
       [3022],
       [3803],
       [3157],
       [3432],
       [2944],
       [3004],
       [2800],
       [2910],
       [3209]])

In [62]:
X_combined = np.concatenate([X_VWAP_scaled, X_Trade_Count], axis=1)

In [63]:
X_combined

array([[2.56852537e-01, 5.04500000e+03],
       [2.57408772e-01, 4.53400000e+03],
       [2.56488985e-01, 4.77500000e+03],
       [2.56084617e-01, 4.23400000e+03],
       [2.55880303e-01, 4.73800000e+03],
       [2.56370740e-01, 4.10000000e+03],
       [2.56226171e-01, 3.18400000e+03],
       [2.56624056e-01, 4.15300000e+03],
       [2.57249550e-01, 3.98500000e+03],
       [2.56798249e-01, 3.09600000e+03],
       [2.56149887e-01, 4.29200000e+03],
       [2.55608589e-01, 3.60600000e+03],
       [2.55537957e-01, 4.31700000e+03],
       [2.56239927e-01, 3.82500000e+03],
       [2.56077761e-01, 2.64400000e+03],
       [2.55733474e-01, 2.71300000e+03],
       [2.55519244e-01, 2.91200000e+03],
       [2.55532020e-01, 3.88700000e+03],
       [2.55138762e-01, 3.06200000e+03],
       [2.54960828e-01, 3.94600000e+03],
       [2.55139939e-01, 3.56300000e+03],
       [2.55523887e-01, 3.32100000e+03],
       [2.55292260e-01, 3.89400000e+03],
       [2.55309004e-01, 3.73900000e+03],
       [2.559374

In [64]:
X_Tensor = np.expand_dims(X_combined, axis=0)

In [65]:
X_Tensor

array([[[2.56852537e-01, 5.04500000e+03],
        [2.57408772e-01, 4.53400000e+03],
        [2.56488985e-01, 4.77500000e+03],
        [2.56084617e-01, 4.23400000e+03],
        [2.55880303e-01, 4.73800000e+03],
        [2.56370740e-01, 4.10000000e+03],
        [2.56226171e-01, 3.18400000e+03],
        [2.56624056e-01, 4.15300000e+03],
        [2.57249550e-01, 3.98500000e+03],
        [2.56798249e-01, 3.09600000e+03],
        [2.56149887e-01, 4.29200000e+03],
        [2.55608589e-01, 3.60600000e+03],
        [2.55537957e-01, 4.31700000e+03],
        [2.56239927e-01, 3.82500000e+03],
        [2.56077761e-01, 2.64400000e+03],
        [2.55733474e-01, 2.71300000e+03],
        [2.55519244e-01, 2.91200000e+03],
        [2.55532020e-01, 3.88700000e+03],
        [2.55138762e-01, 3.06200000e+03],
        [2.54960828e-01, 3.94600000e+03],
        [2.55139939e-01, 3.56300000e+03],
        [2.55523887e-01, 3.32100000e+03],
        [2.55292260e-01, 3.89400000e+03],
        [2.55309004e-01, 3.7390000

In [66]:
model = load_model("/content/drive/MyDrive/AI Financial Analisys/Summer Project/Live Trading/ds=sip+s=AAPL+mp=False+sd=2016-01-1+ed=2024-12-30+tf=5Min+fm=vwap+sm=trade_count+tm=+r=36+sort=False+rfm=False+rsm=False+rtm=False+d=+st=minmax+cts=[0]+Lb=True+e=500+es=True+cb=val_accuracy+p=100+bs=128+tl=0.41606152057647705+ta=0.8365758657455444.keras")

In [67]:
predictions = model.predict(X_Tensor)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 651ms/step


In [68]:
predictions

array([[0.24083064]], dtype=float32)

In [69]:
decisive_sensibility = 0.5

predicted_classes = (predictions >= decisive_sensibility).astype(int)

In [70]:
print(predictions)
print(predicted_classes)

[[0.24083064]]
[[0]]


In [71]:
def add_hours_skipping_night(start: pd.Timestamp, hours: float) -> pd.Timestamp:
    """
    Add `hours` to `start`, but never count any time between 00:00 and 08:00.
    """
    current = start
    remaining = hours
    while remaining > 0:
        # 1) If we're in the blackout (00:00–08:00), jump to 08:00 that same day
        if current.hour < 8:
            current = current.normalize() + timedelta(hours=8)

        # 2) Otherwise, we have until midnight to work with
        end_of_day = current.normalize() + timedelta(days=1)
        available = (end_of_day - current).total_seconds() / 3600.0

        # 3) Consume whichever is smaller: what's left today, or what you still need
        to_add = min(available, remaining)
        current += timedelta(hours=to_add)
        remaining -= to_add

        # 4) If you still have time to add, skip the next 00:00–08:00 and loop
        if remaining > 0:
            current = current.normalize() + timedelta(days=1, hours=8)

    return current

In [ ]:
# --- usage on your DataFrame: ---
last_ts = data.index[-1]        # or data.tail(1).index[0]
end_time_prediction = add_hours_skipping_night(last_ts, 3)

In [73]:
start_str = last_ts.strftime('%b %-d, %Y at %-I:%M %p')       # e.g. “May 9, 2025 at 10:30 PM”
end_str   = end_time_prediction.strftime('%b %-d, %Y at %-I:%M %p')
print(f"Prediction valid from {start_str} until {end_str}.")

Prediction valid from May 9, 2025 at 6:05 PM until May 9, 2025 at 9:05 PM.
